# Reproducing Natraj et al. (2025) from dandiset 001535

**Paper.** Natraj N, Seko S, Abiri R, Miao R, Yan H, Graham Y, Tu-Chan A, Chang EF, Ganguly K
(2025). Sampling representational plasticity of simple imagined movements across days enables
long-term neuroprosthetic control. *Cell* 188, 1208-1225.
[doi:10.1016/j.cell.2025.02.001](https://doi.org/10.1016/j.cell.2025.02.001). The claims file was
written from the accepted author manuscript in PubMed Central (PMC11932800), with the
supplementary legends; it is not the version of record.

**Dandiset.** [DANDI:001535](https://dandiarchive.org/dandiset/001535/0.260608.0617), version
0.260608.0617. The paper cites no version of the dandiset (its data statement points to OSF for
code and to the lead contact for data). There are two published versions, 0.250714.1218 and
0.260608.0617. Both list the same single asset (`sub-BRAVO1/sub-BRAVO1.nwb`, asset
fa402651-4564-4d67-bb11-941843453978, the same blob, 4773567412 bytes); they differ only in
metadata (a second contributor and the citation), none of which the analysis uses. The most recent
published version is pinned, as the protocol asks when the paper cites none.

**What the dandiset holds.** One NWB file for participant B1: 9737 trials of 256 features at
100 Hz (high-gamma amplitude and low-frequency signals on the 128 channels, the input of the
paper's biLSTM decoder), with a target id and two context flags per trial. It holds no data from
participant B3, no Figure 1 imagined-movement task, no delta, beta and high-gamma band-power
features of the MLP decoders, no session, day, open-loop or closed-loop labels, no online decoder
outputs and no robot kinematics or task outcomes.

| Claims | Tested or blocked |
|---|---|
| rnn-training-data-amount | Tested |
| Figure 1 and S2: fig1g-somatotopy-b1, fig1h-somatotopy-b3, fig1ij-within-effector-levels, figs2-pairwise-distances-significant, fig1kl-b1-b3-correlation, fig1mn-hierarchical-clustering (6) | Blocked, data missing (the Figure 1 task and B3 are not in the file) |
| Figures 2, 3, S4 to S6: fig2-session-trial-counts, fig2e-above-chance, fig2e-cl2-vs-ol-accuracy, fig2e-cl1-vs-ol-accuracy, fig2e-cl2-vs-cl1-accuracy, figs4ef-ol-accuracy-stable, figs4g-percent-change-accuracy, figs4hi-feature-importance, fig2i-distance-cl-vs-ol, fig2i-distance-cl2-vs-cl1, fig2j-variance-cl-vs-ol, fig2j-variance-cl2-vs-cl1, figs5-channel-variance-reduction, figs5bc-somatotopy-preserved, figs6-nine-actions, fig2kl-ol-distance-stable, fig2kl-cl-distance-increase, figs6gh-highdim-trends, fig2mn-distance-predicts-accuracy, fig3c-manifold-stable (20) | Blocked, data missing (the delta, beta and high-gamma band power of the MLP decoders, session and OL, CL1, CL2 labels, and B3) |
| Figure S3: figs3-spatial-pooling | Blocked, metadata insufficient (no grid positions for the channels); a variant under assumed layouts is shown below |
| Figure S7: figs7b-refit-less-discernible, figs7cd-refit-less-stable, figs7i-ibid-straighter-than-refit (3) | Blocked, data missing (no ReFit-KF decodes or kinematics) |
| Figure 4: fig4b-day-differences | Blocked, metadata insufficient (no day labels) |
| Figure 4 and S8A-B: fig4cd-day-decodable, fig4cd-demeaned-chance, figs8ab-day-decodable-trials, fig4fg-sampling-days-increases-distance, fig4i-boundary-convergence (5) | Blocked, data missing (B3, or the band-power features, and no day labels) |
| Figure 5 and S8C-E: fig5abc-pnp1-b1, fig5def-pnp2-b1, fig5-pnp-experiments-equal, figs8cde-pnp-stable, fig5-pnp-b1-overall-bitrate, fig5ghi-pnp-b3 (6) | Blocked, data missing (no online decoder outputs or task outcomes) |
| Figure 6 and S8G-I: fig6g-ibid-angular-error, fig6g-velocity-covariation, figs8hi-perspective-perturbation, figs8g-coactivation (4) | Blocked, data missing (no kinematics, perturbation or coactivation labels) |
| Figure 7: fig7de-mode-switch-learned, fig7ij-reach-rotate, fig7k-wall-first-half, fig7k-wall-full-decline, fig7kl-recalibration-restores (5) | Blocked, data missing (no robot task outcomes or times) |

The reason for each blocked claim is in the attempt record; the data check below prints what the
file holds, which is what the blocks rest on.

In [ ]:
import re
import time
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import bci

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

started = time.time()
url = bci.asset_url()
meta, table = bci.file_metadata(url)
right_thumb = table.id[table.target_id == 1].astype(int)   # target 1 is the right thumb (figs3 section)
# Listing the acquisition group takes minutes; it runs alongside the trial summaries.
with ProcessPoolExecutor(1) as lister:
    names_job = lister.submit(bci.acquisition_names, url)
    summ = bci.trial_summaries(url, table.id.astype(int), keep_binned=right_thumb)
    acq_names = names_job.result()
print(f"dandiset {bci.DANDISET_ID} version {bci.DANDISET_VERSION}, {bci.ASSET_PATH}")
print(f"read the metadata and {len(summ)} trials in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
for k in ["identifier", "session_description", "session_start_time", "related_publications", "subject"]:
    print(f"{k}: {meta[k]}")
print("top-level groups:", meta["top_level"])
print("processing modules:", meta["processing"], "; analysis:", meta["analysis"],
      "; electrodes table present:", meta["electrodes"])
print("trial table columns:", meta["trial_columns"])
print("trial series attributes:", meta["trial_series_attrs"], "; unit:", meta["trial_series_unit"])
print(f"\ntrials in table: {len(table)}; ids {table.id.min()} to {table.id.max()}, unique: {table.id.is_unique}")
print("every table id has a series acquisition/trial_<id>:", len(summ) == len(table) and (summ.id.values == table.id.values).all())
print("a series after the last id exists:", meta["series_after_last_id"])
kinds = pd.Series([re.sub(r"\d+", "<n>", n) for n in acq_names]).value_counts()
print("objects in acquisition:", len(acq_names), "; by name pattern:", kinds.to_dict())
print("acquisition holds exactly trial_<id> for each table id and nothing else:",
      sorted(acq_names) == sorted(f"trial_{int(i)}" for i in table.id))
print("start_time equals id:", bool((table.start_time == table.id).all()),
      "; stop_time equals id + 1:", bool((table.stop_time == table.id + 1).all()))
print("\ntrials by target_id and context:")
ctx = np.select([table.is_virtual & ~table.is_robot, ~table.is_virtual & table.is_robot,
                 table.is_virtual & table.is_robot], ["virtual", "robot", "both"], "neither")
table["context"] = ctx
print(pd.crosstab(table.target_id.astype(int), table.context, margins=True).to_string())

The file holds one subject, BRAVO1, which is the paper's B1 (male; the paper gives 41 years at
implant, the file P38Y; neither says at what date the age applies, so this is left open). The
acquisition group holds 9737 objects, all named `trial_<n>`, one for each id of the trial table and
nothing else. No participant B3 is in the dandiset. The trial table's start and stop
times are placeholders (the trial id and the id plus one, as their descriptions say, "DUMMY"), and
the session start time is a single date for all trials: nothing in the file gives a trial's day,
session, block, or whether it was open-loop or closed-loop, and there is no electrodes table,
so the channels' places on the 8x16 grid are not given. The dandiset description speaks of 7
directional targets; the table has 9 target ids, 1 to 7 with about 1350 trials each and 8 and 9
with 132 each. Nothing in the file names the targets.

In [ ]:
dur = summ.n_samples / bci.SAMPLING_HZ
print(f"features per sample: {sorted(summ.n_features.unique())}")
print(f"timestamps start at 0 in every trial: {bool((summ.t_first == 0).all())}; "
      f"step from {summ.dt_min.min():.6f} to {summ.dt_max.max():.6f} s (description: 100 Hz)")
print(f"samples per trial: {summ.n_samples.min()} to {summ.n_samples.max()}, median {summ.n_samples.median():.0f}")
print(f"last timestamp equals (samples - 1) / 100 s: {bool(np.allclose(summ.t_last, (summ.n_samples - 1) / 100))}")
print(f"NaN values: {summ.n_nan.sum()}")
print(f"L2 norm of columns 0-127 per trial: {summ.norm_a.min():.6f} to {summ.norm_a.max():.6f}")
print(f"L2 norm of columns 128-255 per trial: {summ.norm_b.min():.6f} to {summ.norm_b.max():.6f}")
print(f"smallest value in columns 0-127 over all trials: {summ.min_a.min():.3g}; in columns 128-255: {summ.min_b.min():.3g}")
print(f"trials with a negative value in columns 0-127: {(summ.min_a < 0).sum()} of {len(summ)}; "
      f"in columns 128-255: {(summ.min_b < 0).sum()}")
print(f"largest absolute value over all trials: {summ.abs_max.max():.3g}")
dup = summ[summ.duplicated("hash", keep=False)].sort_values("hash")
print(f"\ntrials whose data are byte-identical to another trial: {len(dup)}")
if len(dup):
    print(dup.groupby("hash").id.apply(list).to_string())
means = np.round(np.vstack(summ["mean"].values), 7)
_, inv, counts = np.unique(means, axis=0, return_inverse=True, return_counts=True)
print(f"trials sharing their 256 time-averaged feature values with another trial: {(counts[inv] > 1).sum()}")

Every trial has 256 features sampled every 0.01 s, as described (100 Hz), and lasts 1.8 to
11.1 s. The first 128 columns have no negative value in 9239 of the 9737 trials (the smallest value
is -0.015) and the last 128 take both signs in every trial, which fits high-gamma amplitude followed by the low-frequency signal; the description names both but not the order, so this order
is consistent with the data, not shown. Each block of 128 columns has an L2 norm of 1 over the
whole trial, which is what the Methods (Spatiotemporal feature extraction for RNNs) describe for
the biLSTM input: "each feature data matrix was transformed to have an L 2 norm of 1 and
concatenated together". The values are therefore not in physical units and are scaled per trial;
the unit "arbitrary" says so. The duplicate checks above count trials deposited more than once. They find exact copies, and copies whose time-averaged features agree to 7 decimals; a trial deposited twice with different trimming would have a different length and L2 scaling, and these checks would not catch it.

### Do the trial labels belong to the trial data?

There are no event times inside a trial and no session clock, so the check is that each trial's
target id belongs to the series of the same id. Each trial is reduced to its 256 time-averaged
features and the target is decoded by cross-validated logistic regression (5 stratified folds, a
standard choice). The contrast is the same decoder with the labels shifted by one trial, and
with the labels shuffled.

In [ ]:
rng = np.random.default_rng(0)
X = np.vstack(summ["mean"].values).astype(float)
y = table.target_id.astype(int).values
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
cv = StratifiedKFold(5, shuffle=True, random_state=0)
rows = []
for name, labels in [("true labels", y), ("shifted by +1 trial", np.roll(y, 1)),
                     ("shifted by -1 trial", np.roll(y, -1)), ("shuffled", rng.permutation(y))]:
    acc = cross_val_score(clf, X, labels, cv=cv)
    rows.append({"labels": name, "accuracy": acc.mean(), "fold_min": acc.min(), "fold_max": acc.max()})
chance = pd.Series(y).value_counts(normalize=True).max()
print(pd.DataFrame(rows).round(4).to_string(index=False))
print(f"chance (largest class share): {chance:.4f}; 1/9 = {1/9:.4f}")
by_ctx = []
for c in ["virtual", "robot"]:
    m = (table.context == c).values
    acc = cross_val_score(clf, X[m], y[m], cv=cv).mean()
    by_ctx.append({"context": c, "trials": int(m.sum()), "accuracy": round(acc, 4)})
print(pd.DataFrame(by_ctx).to_string(index=False))

With the true labels the target is decoded from the trials' own features with 0.72 accuracy
(0.72 in virtual and 0.76 in robot trials), against 0.17 to 0.18 with the labels moved by one
trial, 0.13 with them shuffled, and 0.14 for always naming the largest class. The target ids are
consistent with the series they share an id with, and not with the neighbouring trials' series. This shows the table and the series are
aligned trial by trial; it cannot show anything about timing within a trial, as the file has no
event times. The labels moved by one trial still score 0.175, above the 0.139 of the largest
class. The trial order is not random (a trial's next trial has the same target less often than in
a random order of each context, as printed in the next section), so the order of trials carries
some information about a neighbour's target; this does not weaken the alignment shown by the drop
from 0.72. The folds mix trials of all sessions, which is acceptable for an alignment check.

## rnn-training-data-amount

> The biLSTM decoder was trained on about 14 hours and 48 minutes of open-loop and closed-loop
> data from B1's virtual and real-world hDoF experiments, before augmentation and segmentation.

Expected 888 min, tolerance 88.8 min. The criterion: the total duration of the trials used to
train the RNN, before segmentation; reading (a) the whole stored trial, reading (b) the active
portion only; shown with and without the real-robot trials when the dataset labels contexts.

A trial's duration is its number of samples divided by 100 Hz. The dandiset's description says
its trials are the ones "uploaded for the development and testing of novel BCI decoding
architectures" and points to the paper's biLSTM; it does not say that they are exactly the
paper's training set. Reading (b) cannot be run, because the file has no task states, but the
active portion of a trial is part of the stored trial, so reading (b) is at most reading (a).
Below, the durations are broken down by context and target, the order of contexts in trial-id
order is shown, and the high-gamma time course at the start and end of each trial is shown by
context, to see whether the stored trials hold a pre-cue baseline or are already the active
epochs that the OSF code (get_lstm_features.m, task states 2 and 3 plus padding) extracts.

Which robot trials the paper counts is open. The 14 h 48 min paragraph counts data "in the virtual
3D environment and with practice with the Kinova Jaco in the real-world using the MLP"; the same
paragraph says "We split the collected data from the virtual environment into 85% training and 15%
validation when training the RNN", and the RNN was then fine-tuned on "multiple blocks of
open-loop and closed-loop center-out and stop data with the physical robot". If the 1886 robot
trials are that fine-tuning data, they are not in the 14 h 48 min.

In [ ]:
dur = summ.n_samples / bci.SAMPLING_HZ
d = table.assign(duration_s=dur.values)
rows = [{"trials": "all", "n": len(d), "total_min": d.duration_s.sum() / 60}]
for c in ["virtual", "robot"]:
    s = d[d.context == c]
    rows.append({"trials": c, "n": len(s), "total_min": s.duration_s.sum() / 60})
s = d[d.target_id <= 7]
rows.append({"trials": "targets 1 to 7 only", "n": len(s), "total_min": s.duration_s.sum() / 60})
alt = ((summ.n_samples - 1) / bci.SAMPLING_HZ).sum() / 60
rows.append({"trials": "all, last minus first timestamp", "n": len(d), "total_min": alt})
res = pd.DataFrame(rows)
res["total_h_min"] = res.total_min.map(lambda m: f"{int(m // 60)} h {m % 60:.1f} min")
res["within_888_pm_88.8"] = (res.total_min - 888).abs() <= 88.8
print(res.round(1).to_string(index=False))
print(f"\nexpected 888 min; the shortest duration that would match is {888 - 88.8:.1f} min")
print(f"trial duration: {d.duration_s.min():.2f} to {d.duration_s.max():.2f} s, median {d.duration_s.median():.2f} s")
print(d.groupby("context").duration_s.describe().round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
for c in ["virtual", "robot"]:
    ax.hist(d.duration_s[d.context == c], bins=np.arange(0, 12.01, 0.1), alpha=0.6, label=c)
ax.set_xlabel("trial duration (s)")
ax.set_ylabel("trials")
ax.legend()
plt.tight_layout()
plt.show()
print(f"all {len(d)} trials are inside the axes (0 to 12 s); longest {d.duration_s.max():.2f} s")

In [ ]:
by = d.groupby(["context", d.target_id.astype(int)]).duration_s.agg(["size", "sum", "median", "min", "max"])
by["sum"] = by["sum"] / 60
by.columns = ["trials", "total_min", "median_s", "min_s", "max_s"]
print(by.round(2).to_string())

In [ ]:
runs = d.groupby((d.context != d.context.shift()).cumsum()).agg(
    context=("context", "first"), first_id=("id", "min"), last_id=("id", "max"), trials=("id", "size"))
print("runs of one context in trial-id order:")
print(runs.to_string(index=False))
for k in (8, 9):
    ids = d.id[d.target_id == k]
    print(f"target {k}: ids {int(ids.min())} to {int(ids.max())}, {len(ids)} trials")
trun = d.groupby((d.target_id != d.target_id.shift()).cumsum()).size()
print(f"runs of one target in trial-id order: {len(trun)}, longest {trun.max()}, "
      f"share of trials whose next trial has the same target: {(d.target_id.values[1:] == d.target_id.values[:-1]).mean():.3f}")
p_ctx = d.groupby("context").target_id.value_counts(normalize=True)
same_random = sum((d.context == c).mean() * (p_ctx[c] ** 2).sum() for c in ["virtual", "robot"])
print(f"expected share if the trials of each context were in random order: {same_random:.3f}")

In [ ]:
# Channel-averaged high gamma, rescaled per trial so that its root mean square over the trial is 1
# (bci docstring); the trial mean is the reference level.
hg_mean = np.vstack(summ["mean"].values)[:, :bci.N_CHANNELS].mean(axis=1) * np.sqrt(summ.n_samples * bci.N_CHANNELS)
S = np.vstack(summ["hg_start"].values)
E = np.vstack(summ["hg_end"].values)
t = np.arange(bci.EDGE) / bci.SAMPLING_HZ
fig, axs = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
rows = []
lo, hi = np.inf, -np.inf
for c in ["virtual", "robot"]:
    m = (table.context == c).values
    s, e = S[m].mean(axis=0), E[m].mean(axis=0)
    lo, hi = min(lo, s.min(), e.min()), max(hi, s.max(), e.max())
    axs[0].plot(t, s, label=c)
    axs[1].plot(t - t[-1], e, label=c)
    first = S[m, :50].mean(axis=1)
    rows.append({"context": c, "trials": int(m.sum()), "trial mean": hg_mean[m].mean(),
                 "first 0.5 s": first.mean(), "0.5 to 1 s": S[m, 50:].mean(),
                 "last 1 s": E[m].mean(),
                 "share of trials with first 0.5 s below trial mean": (first < hg_mean[m]).mean(),
                 "median ratio first 0.5 s / trial mean": np.median(first / hg_mean[m])})
axs[0].set_xlabel("time from trial start (s)")
axs[1].set_xlabel("time from trial end (s)")
axs[0].set_ylabel("channel-mean high gamma (trial RMS = 1)")
axs[0].legend()
plt.tight_layout()
plt.show()
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"plotted means range from {lo:.3f} to {hi:.3f}; all inside the axes")
print("\nmean at the start, in 0.1 s steps from 0 s:")
for c in ["virtual", "robot"]:
    m = (table.context == c).values
    print(f"  {c}: " + " ".join(f"{v:.3f}" for v in S[m].mean(axis=0)[::10]))
for cut in (0.5, 1.0):
    n_virtual = (d.context == "virtual").sum()
    print(f"reading (a) less the first {cut} s of every virtual trial: {d.duration_s.sum() / 60 - n_virtual * cut / 60:.1f} min; "
          f"less the first {cut} s of every trial: {d.duration_s.sum() / 60 - len(d) * cut / 60:.1f} min (floor 799.2)")

At their start the virtual trials' channel-mean high gamma is lower than over the whole
trial: it stays at 0.830 to 0.843 for the first 0.4 s and rises to 0.926 by 0.7 s (first 0.5 s
0.852 against a trial mean of 0.877; lower in 69% of virtual trials, median ratio 0.967). The
robot trials show no such rise (0.888 to 0.903 over the first second; first 0.5 s lower than the
trial mean in 37%). Both contexts end at about the same level (0.862 and 0.860 over the last
second). The virtual trials may therefore begin with about 0.4 s before activity rises, which
could be a short period before the cue or the time the participant takes to respond; these outputs
cannot tell which, as the file has no task states. If the low start is reaction time after the cue, it lies inside the active (cue and
control) portion and reading (b) keeps it. In the worst case, if the first 0.5 s of every virtual
trial were outside the active portion, reading (b) would be 771.9 min, below the 799.2 min floor;
this is a lower bound, not an estimate. Trial lengths are not random: robot trials last 6.0 to 11.1 s with a median of 9.01 s in
every target, and every target has about the same robot total (42.2 to 43.0 min). In trial-id
order all 7851 virtual trials come first (ids 1 to 7851) and all 1886 robot trials after (ids
7852 to 9737); targets 8 and 9 are spread through the virtual ids (12 to 7777 and 156 to 7826). The
file does not say whether ids follow the recording order, so this order cannot tell whether the
robot trials are the later fine-tuning data.

**Result: reproduced.** Under reading (a), the whole stored trial, the 9737 trials last 837.4 min
(13 h 57.4 min), within the 888 +/- 88.8 min of the claim (from 799.2 min); without targets 8 and
9 they last 814.3 min, and as last minus first timestamp 835.7 min, both also within. Reading (b),
the active portion, cannot be run, as the file has no task states; it can only be shorter than (a),
and the time course above does not show whether the stored trials already are the active epochs.
In the worst case, if the first 0.5 s of the virtual trials, where their high gamma has not yet
risen, were outside the active portion, reading (b) would be 771.9 min and outside the tolerance;
the outputs cannot tell whether that time is before the cue or reaction time after it, so this is
a lower bound, not an estimate. The criterion asks for a match under one reading, which (a) gives;
whether the readings should have to agree is asked of a reader.
The match rests on counting the robot trials: the 7851 virtual trials alone last 540.0 min, outside
the tolerance, and the 1886 robot trials 297.3 min. The paper counts its 14 h 48 min over the virtual
environment and "practice with the Kinova Jaco in the real-world using the MLP", so the robot trials
belong in the total if they are that practice; if they are the later fine-tuning data of the RNN,
the total is 540.0 min and the claim would not be reproduced. The file does not say which, and the
robot trials' place after all virtual trials in id order does not settle it. The outcome rests on
the assumption that they are the MLP practice.

## figs3-spatial-pooling (blocked; a variant under assumed layouts)

> In B1, 2x2 spatial pooling of the grid makes single-trial high-gamma activity more consistent
> and improves online decoding in pilot blocks.

The criterion: variance over time and trials of binned (200 ms) high gamma during right thumb
trials, per channel, for the 128 raw channels against the 32 channels after a 1 s running average
and 2x2 pooling with stride 2; Mann Whitney U; reproduced when p < 0.05 with pooled lower.

The claim is blocked as metadata insufficient: the file has no electrodes table, so the place of
each channel on the 8x16 grid, which the 2x2 pooling needs, is not given. Two other differences
stand: the stored high gamma is the biLSTM's amplitude at 100 Hz, L2 normalized per trial, not the
MLP pipeline's 200 ms band power, and there are no session labels, so all right thumb trials are
used where the paper used one session. Target 1 is the right thumb under both the Methods' map
(X+) and the OSF code's names (RealRobotBatch/RunTrial.m), as the claims file's header gives them.

The variant below bins the stored high gamma into 200 ms bins, applies a running average over 5
bins within each trial, and pools 2x2 under two assumed layouts (channels in row-major and in
column-major order on the 8x16 grid) and under 200 random layouts. Averaging lowers variance
whatever the layout, so the random layouts show how much of the result is mechanical. The
accuracy part needs online pilot blocks, which the file does not hold.

In [ ]:
def mw(a, b):
    """Mann Whitney U of a against b, with the normal-approximation z."""
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    n1, n2 = len(a), len(b)
    return (u - n1 * n2 / 2) / np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12), p

rt = summ[summ.id.isin(right_thumb)]
binned = list(rt.binned)
rt_ctx = table.set_index("id").loc[rt.id, "context"].values
print(f"right thumb trials: {len(binned)} ({(rt_ctx == 'virtual').sum()} virtual, {(rt_ctx == 'robot').sum()} robot); "
      f"200 ms bins per trial {min(len(b) for b in binned)} to {max(len(b) for b in binned)}")
raw = np.vstack(binned)
smooth = np.vstack([bci.running_mean(b) for b in binned])
var_raw = raw.var(axis=0)
layouts = {"row-major": np.arange(128).reshape(8, 16), "column-major": np.arange(128).reshape(16, 8).T}
rows = []
for name, lay in layouts.items():
    for label, x in [("1 s average and 2x2 pooling", smooth), ("2x2 pooling only", raw)]:
        v = bci.pool_2x2(x, lay).var(axis=0)
        z, p = mw(var_raw, v)
        rows.append({"layout": name, "pooled set": label, "median var raw": np.median(var_raw),
                     "median var pooled": np.median(v), "ratio": np.median(v) / np.median(var_raw), "z": z, "p": p})
z, p = mw(var_raw, smooth.var(axis=0))
rows.append({"layout": "none", "pooled set": "1 s average only (128 channels)", "median var raw": np.median(var_raw),
             "median var pooled": np.median(smooth.var(axis=0)),
             "ratio": np.median(smooth.var(axis=0)) / np.median(var_raw), "z": z, "p": p})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

rng = np.random.default_rng(15351)
rand = []
for _ in range(200):
    v = bci.pool_2x2(smooth, rng.permutation(128).reshape(8, 16)).var(axis=0)
    z, p = mw(var_raw, v)
    rand.append((np.median(v) / np.median(var_raw), z, p, np.median(v) < np.median(var_raw)))
rand = np.array(rand)
print(f"\n200 random layouts, 1 s average and 2x2 pooling: ratio of medians {rand[:, 0].min():.3f} to {rand[:, 0].max():.3f}, "
      f"z {rand[:, 1].min():.2f} to {rand[:, 1].max():.2f}; pooled lower with p < 0.05 in {int(((rand[:, 2] < 0.05) & (rand[:, 3] == 1)).sum())} of 200")

print("\nby context, row-major layout, 1 s average and 2x2 pooling:")
for c in ["virtual", "robot"]:
    keep = [b for b, cc in zip(binned, rt_ctx) if cc == c]
    vr = np.vstack(keep).var(axis=0)
    vp = bci.pool_2x2(np.vstack([bci.running_mean(b) for b in keep]), layouts["row-major"]).var(axis=0)
    z, p = mw(vr, vp)
    print(f"  {c}: {len(keep)} trials, median var raw {np.median(vr):.3g}, pooled {np.median(vp):.3g}, z = {z:.2f}, p = {p:.3g}")

**Result: blocked (metadata insufficient).** Under the variant, 2x2 pooling after a 1 s running
average lowers the median per-channel variance of the 1357 right thumb trials to 0.499 of the raw
value with channels in row-major order (z = 7.05, p = 1.9e-12) and to 0.538 in column-major order
(z = 7.12, p = 1.1e-12); pooling alone gives ratios of 0.633 and 0.677, and the running average
alone 0.67 (z = 6.74). Every one of 200 random layouts gives the same direction with p < 0.05
(ratios 0.457 to 0.533, z 7.40 to 8.05), so the direction and significance of the result do not
depend on where the channels are on the grid: averaging lowers variance whatever channels are
averaged. The size of the effect does depend on the layout: the column-major ratio (0.538) lies
above every random layout's. It holds in virtual (z =
7.42) and robot (z = 8.58) trials alone. Since it cannot show anything about the grid, and the
features are per-trial L2-normalized amplitude, not the paper's 200 ms band power in one session,
the variant does not decide the claim, which stays blocked for want of the electrode positions.

## Summary

| Claim | Outcome |
|---|---|
| rnn-training-data-amount | Reproduced, resting on the assumption that the robot trials are the MLP practice the paper counts |
| figs3-spatial-pooling | Blocked, metadata insufficient (no grid positions); a variant under assumed layouts lowers the variance under every layout, including random ones |
| fig4b-day-differences | Blocked, metadata insufficient (no day labels) |
| The other 49 claims | Blocked, data missing |

How the analysis differs from the paper's: the one tested claim is a count of stored data, and
the dandiset does not say whether its trials are the whole of the paper's training data or how
much of each trial was kept.

What was found about the dataset:

- It holds only participant B1's biLSTM input features (high gamma and low frequency, 100 Hz, L2
  normalized per trial), with a target id and a virtual or robot flag per trial. Participant B3,
  the Figure 1 task, the band-power features of the MLP decoders, online decoder outputs and robot
  kinematics are not in it.
- No trial has a day, session, block or open-loop or closed-loop label; trial start and stop times
  are placeholders.
- The target ids carry no names, and there are 9 of them where the description says 7 targets.
- There is no electrodes table, so the channels cannot be placed on the grid.
- The robot trials (1886, 6.0 to 11.1 s each, all after the virtual trials in id order) are not
  said to be either the MLP practice the paper counts in its 14 h 48 min or the later fine-tuning
  of the RNN. Without them the stored total is 540.0 min, outside the claim's tolerance.
- Nothing says whether a stored trial is the whole trial or only its active portion.
- No trial is deposited twice as an exact copy (no two trials have identical data or identical
  time-averaged features); a copy with different trimming would not be found by these checks.
- The acquisition group holds only the 9737 trial series, one per trial id.